In [18]:
import pandas as pd
import numpy as np
import csv
 
# ===========================================================
# data_extraction_FINAL.py — CERTIFIED FINAL VERSION
# Verified against raw_data.csv (881,743 rows, 28 columns)
# Expected output: MASTER_Remittance_V7.csv (351,893 records)
# ===========================================================
 
INPUT_FILE  = 'raw_data.csv'
OUTPUT_FILE = 'MASTER_Remittance_V7.csv'
BASE_BUFFER = 12000
 
RPP = {
    1:88.6,  2:105.4, 4:101.3,  5:88.0,  6:115.9, 8:106.8,
    9:116.2,10:109.3,11:122.4, 12:100.3,13:92.8, 15:118.2,
    16:93.3,17:105.4,18:91.5,  19:90.2, 20:90.8, 21:89.6,
    22:90.6,23:99.6, 24:112.2, 25:117.9,26:93.0, 27:99.6,
    28:84.8,29:90.8, 30:95.2,  31:90.4, 32:105.4,33:110.2,
    34:116.3,35:91.8,36:121.4, 37:92.2, 38:93.8, 39:93.2,
    40:88.4,41:105.9,42:101.3, 44:115.2,45:92.2, 46:89.8,
    47:90.2,48:96.0, 49:98.2,  50:105.6,51:103.8,53:110.2,
    54:87.6,55:96.6, 56:94.0
}
 
STATE_MAP = {
    1:'Alabama',2:'Alaska',4:'Arizona',5:'Arkansas',6:'California',
    8:'Colorado',9:'Connecticut',10:'Delaware',11:'DC',
    12:'Florida',13:'Georgia',15:'Hawaii',16:'Idaho',17:'Illinois',
    18:'Indiana',19:'Iowa',20:'Kansas',21:'Kentucky',22:'Louisiana',
    23:'Maine',24:'Maryland',25:'Massachusetts',26:'Michigan',
    27:'Minnesota',28:'Mississippi',29:'Missouri',30:'Montana',
    31:'Nebraska',32:'Nevada',33:'New Hampshire',34:'New Jersey',
    35:'New Mexico',36:'New York',37:'North Carolina',38:'North Dakota',
    39:'Ohio',40:'Oklahoma',41:'Oregon',42:'Pennsylvania',
    44:'Rhode Island',45:'South Carolina',46:'South Dakota',
    47:'Tennessee',48:'Texas',49:'Utah',50:'Vermont',
    51:'Virginia',53:'Washington',54:'West Virginia',
    55:'Wisconsin',56:'Wyoming'
}
 
COUNTRY_DATA = {
    20000:('Mexico',      0.158, 4320),
    52100:('India',       0.091, 6200),
    51500:('Philippines', 0.132, 5100),
    26020:('El Salvador', 0.182, 3900),
    52110:('Bangladesh',  0.163, 3200),
    60053:('Somalia',     0.201, 2800),
}
 
COUNTRY_INCOME_FLOOR = {
    'Mexico':27000,'India':62000,'Philippines':48000,
    'El Salvador':24000,'Bangladesh':31000,'Somalia':22000
}
 
# STEP 1: LOAD
print('Loading raw_data.csv...')
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f'  Loaded: {len(df):,} rows')
required = ['STATEFIP','PUMA','GQ','RELATE','HHINCOME','INCWAGE',
            'OWNCOST','RENTGRS','VEHICLES','AGE','CITIZEN','YRSUSA1','OCC','BPLD']
missing = [c for c in required if c not in df.columns]
if missing: raise ValueError(f'Missing columns: {missing}')
 
# STEP 2: GEOGRAPHY
print('Building geography...')
df['STATE_NAME']          = df['STATEFIP'].map(STATE_MAP).fillna('Unknown')
df['COUNTRY_NAME']        = df['BPLD'].map({k:v[0] for k,v in COUNTRY_DATA.items()}).fillna('Other')
df['COUNTRY_REMIT_SHARE'] = df['BPLD'].map({k:v[1] for k,v in COUNTRY_DATA.items()}).fillna(0.10)
df['COUNTRY_AVG_REMIT']   = df['BPLD'].map({k:v[2] for k,v in COUNTRY_DATA.items()}).fillna(2500)
 
# GEOID: 2-digit state + 5-digit PUMA = 7-digit string (FIX APPLIED)
df['GEOID_JOIN'] = (
    df['STATEFIP'].astype(str).str.zfill(2) +
    df['PUMA'].astype(str).str.zfill(5)
).astype(str).str.zfill(7)
assert (df['GEOID_JOIN'].str.len()==7).all(), 'GEOID error!'
print(f'  GEOID check: all 7-digit. OK.')
 
# STEP 3: FILTERS
print('Filtering...')
n0 = len(df)
df = df[df['GQ'].isin([1,2])]
print(f'  After GQ filter: {len(df):,}')
if 'RELATE' not in df.columns:
    raise ValueError('RELATE missing — re-download IPUMS with RELATE variable')
df = df[df['RELATE']==1]
print(f'  After RELATE==1: {len(df):,}')
df = df[df['AGE']>=18]
print(f'  After AGE>=18:   {len(df):,}  (expected 351,893)')
 
# STEP 4: CLEAN N/A CODES
print('Cleaning N/A codes...')
df['HHINCOME'] = df['HHINCOME'].replace([9999999],0)
df.loc[df['HHINCOME']<0,'HHINCOME'] = 0
df['INCWAGE']  = df['INCWAGE'].replace([999999],0)
df['OWNCOST']  = df['OWNCOST'].replace([99999],0)
df['RENTGRS']  = df['RENTGRS'].fillna(0)
df['VEHICLES'] = df['VEHICLES'].replace(9, np.nan)
df['VEHICLES'] = df['VEHICLES'].fillna(df['VEHICLES'].median())
df['YRSUSA1']  = pd.to_numeric(df['YRSUSA1'].astype(str).str.strip(),
                               errors='coerce').fillna(0).astype(int)
 
# STEP 5: FEATURE ENGINEERING
print('Engineering features...')
df['MONTHLY_HOUSING_COST'] = df['RENTGRS'] + df['OWNCOST']
df['ANNUAL_SURVIVAL_COST'] = df['MONTHLY_HOUSING_COST'] * 12
df['RPP']             = df['STATEFIP'].map(RPP).fillna(100.0)
df['BUFFER_ADJUSTED'] = BASE_BUFFER * df['RPP'] / 100
df['SHADOW_LIQUIDITY_GAP'] = df['ANNUAL_SURVIVAL_COST'] - df['HHINCOME']
df['IS_SHADOW_ACTOR'] = (
    (df['HHINCOME'] - df['ANNUAL_SURVIVAL_COST']) < df['BUFFER_ADJUSTED']
).astype(int)
df['IS_EMPLOYED']     = (df['OCC'] > 0).astype(int)
df['IS_CITIZEN']      = df['CITIZEN'].isin([1,2]).astype(int)
df['RECENT_ARRIVAL']  = (df['YRSUSA1'] <= 3).astype(int)
df['MONTHLY_INCOME']  = df['HHINCOME'] / 12
country_order = ['Mexico','India','Philippines','El Salvador','Bangladesh','Somalia','Other']
df['COUNTRY_ENCODED'] = df['COUNTRY_NAME'].apply(
    lambda x: country_order.index(x) if x in country_order else len(country_order))
 
# Remittance estimate
df['COUNTRY_ALPHA'] = df['COUNTRY_NAME'].map({
    'Mexico':0.158,'India':0.091,'Philippines':0.132,
    'El Salvador':0.182,'Bangladesh':0.163,'Somalia':0.201}).fillna(0.10)
df['LIKELY_REMITTER'] = (
    (df['IS_SHADOW_ACTOR']==1)|(df['RECENT_ARRIVAL']==1)|(df['IS_CITIZEN']==0)
).astype(int)
df['INCOME_FLOOR']     = df['COUNTRY_NAME'].map(COUNTRY_INCOME_FLOOR).fillna(25000)
df['INCOME_FOR_REMIT'] = df[['HHINCOME','INCOME_FLOOR']].max(axis=1)
df['EST_ANNUAL_REMITTANCE'] = (
    df['LIKELY_REMITTER'] * df['COUNTRY_ALPHA'] * df['INCOME_FOR_REMIT']
).clip(lower=0)
 
# STEP 6: QUALITY REPORT
print()
print('='*60)
print('DATA QUALITY REPORT')
print('='*60)
print(f'Records:          {len(df):,}  (expected 351,893)')
print(f'GEOID errors:     {(df["GEOID_JOIN"].str.len()!=7).sum()}  (must be 0)')
print(f'AGE<18:           {(df["AGE"]<18).sum()}  (must be 0)')
print(f'VEHICLES null:    {df["VEHICLES"].isnull().sum()}  (must be 0)')
print(f'IS_SHADOW rate:   {df["IS_SHADOW_ACTOR"].mean()*100:.2f}%')
print(f'EST remittance:   ${df["EST_ANNUAL_REMITTANCE"].sum()/1e9:.3f}B')
print(f'National x4.3:    ${df["EST_ANNUAL_REMITTANCE"].sum()*4.3/1e9:.2f}B')
print('='*60)
 
# STEP 7: EXPORT (GEOID FIX APPLIED)
print('Exporting...')
DROP = ['YEAR','SAMPLE','SERIAL','CBSERIAL','CLUSTER','STRATA','GQ',
        'COSTELEC','COSTGAS','COSTWATR','HHWT','PERNUM','PERWT',
        'BPL','STATEFIP','BPLD','RELATE','RELATED']
df_out = df.drop(columns=[c for c in DROP if c in df.columns])
df_out['GEOID_JOIN'] = df_out['GEOID_JOIN'].astype(str).str.zfill(7)
df_out.to_csv(OUTPUT_FILE, index=False, quoting=csv.QUOTE_NONNUMERIC)
verify = pd.read_csv(OUTPUT_FILE, dtype={'GEOID_JOIN':str}, nrows=200)
assert (verify['GEOID_JOIN'].str.len()==7).all(), 'GEOID save verification failed!'
print(f'SUCCESS: {OUTPUT_FILE}  ({len(df_out):,} records)')
print(f'GEOID post-save check: PASS')
print('Next: Run ai_model_FINAL.py')

Loading raw_data.csv...
  Loaded: 881,743 rows
Building geography...
  GEOID check: all 7-digit. OK.
Filtering...
  After GQ filter: 854,920
  After RELATE==1: 351,924
  After AGE>=18:   351,893  (expected 351,893)
Cleaning N/A codes...
Engineering features...

DATA QUALITY REPORT
Records:          351,893  (expected 351,893)
GEOID errors:     0  (must be 0)
AGE<18:           0  (must be 0)
VEHICLES null:    0  (must be 0)
IS_SHADOW rate:   15.27%
EST remittance:   $2.041B
National x4.3:    $8.78B
Exporting...
SUCCESS: MASTER_Remittance_V7.csv  (351,893 records)
GEOID post-save check: PASS
Next: Run ai_model_FINAL.py
